In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import Pipeline
music_df = pd.read_csv('spotify_songs_dataset.csv')

In [ ]:
print(music_df[['duration_ms',"loudness","speechiness"]].describe())
music_df.columns

Index(['song_id', 'song_title', 'artist', 'album', 'genre', 'release_date',
       'duration', 'popularity', 'stream', 'language', 'explicit_content',
       'label', 'composer', 'producer', 'collaboration'],
      dtype='str')

### Why scale our data

- Many models use some form of distace to inform them
- Feature on larger scales can disproportionatelt influence the model
- Example : KNN uses distance explicitly when making predictions
- We want features to be on similar scale
- Normalizing or standardizing (scaling and centering)

### How to scale our data
- Subtract the mean and divide by variance 
    - All features are centered around zero and have a variance of one
    - This is called Standardization
- Can Also subtract the maximum and divide by the range
    - minimum zero and maximum one
- Cann also normalize so the data ranges from +1 to -1


In [ ]:
from sklearn.preprocessing import StandardScaler
drop_cols = [
    'track_artist',
    'track_name',
    'track_album_name',
    'playlist_name',
    'track_href',
    'uri',
    'analysis_url',
    'track_id',
    'id',
    'playlist_id',
    'type',
    'track_artist',
    'track_album_name',
    'playlist_name',
    'track_href',
    'uri',
    'track_album_release_date',
    'track_album_id',
    'playlist_subgenre',
    'playlist_genre'
]

music_clean = music_df.drop(columns=drop_cols)

X = music_clean.drop("track_popularity",axis=1).values
y = music_clean['track_popularity'].values


X_train , X_test , y_train , y_test = train_test_split(X,y,test_size=0.2,random_state=42)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(np.mean(X),np.std(X))
print(np.mean(X_train_scaled),np.std(X_train_scaled))



16514.51828118314 59413.52271446916
-8.59592901056501e-16 1.0000000000000002


# Scalar in a pipeline

In [34]:
steps = [('scalar',StandardScaler()),
         ('knn',KNeighborsClassifier(n_neighbors=6))]

pipeline = Pipeline(steps)
X_train , X_test , y_train , y_test = train_test_split(X,y,test_size=0.2,random_state=42)

knn_scaled = pipeline.fit(X_train,y_train)
y_pred = knn_scaled.predict(X_test)
print(knn_scaled.score(X_test,y_test))

0.10946745562130178


# Comparing performance using unscaled data

In [35]:
X_train , X_test , y_train , y_test = train_test_split(X,y,test_size=0.2,random_state=42)
knn_unscaled = KNeighborsClassifier(n_neighbors=6).fit(X_train,y_train)
print(knn_unscaled.score(X_test,y_test))

0.07988165680473373


# Cross Validation and Scalling in a pipeline

In [38]:
from sklearn.model_selection import GridSearchCV
steps = [('scalar',StandardScaler()),
         ('knn',KNeighborsClassifier())]
pipeline = Pipeline(steps)   
parameters = {"knn__n_neighbors" : np.arange(1,50)}

X_train , X_test , y_train , y_test = train_test_split(X,y,test_size=0.2,random_state=42)

cv = GridSearchCV(pipeline,param_grid=parameters)
cv.fit(X_train,y_train)
y_pred = cv.predict(X_test)

print(cv.best_score_)
print(cv.best_params_)


f:\mubashir\Coding\Machine Learning\venv\Lib\site-packages\sklearn\model_selection\_split.py:813: UserWarning: The least populated class in y has only 1 members, which is less than n_splits=5.
  warnings.warn(


0.22701363073110287
{'knn__n_neighbors': np.int64(1)}
